In [ ]:
# https://pyimagesearch.com/2021/11/01/training-an-object-detector-from-scratch-in-pytorch/

In [2]:
# Define Imports

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Set random seed for reproducibility
seed = 42
torch.manual_seed(seed)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set pin memory for DataLoader
PIN_MEMORY = True if DEVICE.type == 'cuda' else False

Using device: cuda


In [3]:
# Set hyperparameters
LR = 0.001  # Learning rate
BATCH_SIZE = 32  # Batch size
EPOCHS = 10  # Number of epochs


# Set ImageNet mean and std
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Set loss weights
LABEL_WEIGHTS = 1.0
BBOX_WEIGHTS = 1.0

In [4]:
# Create custom dataset class
class ValorantTensorDataset(Dataset):
    def __init__(self, tensors, transform=None):
        self.tensors = tensors
        self.transform = transform

    def __getitem__(self, index):
        # get the image, label, and bounding box tensors
        image = self.tensors[0][index]
        label = self.tensors[1][index]
        bbox = self.tensors[2][index]

        # transpose the image tensor to match the expected input shape
        image = image.permute(2, 0, 1)

        # apply transformations if specified
        if self.transform:
            image = self.transform(image)

        return (image, label, bbox)

    def __len__(self):
        return self.tensors[0].size(0)

In [5]:
train = pd.read_csv('train.csv')
train.head()

,filename,filepath,width,height,depth,label,xmin,ymin,xmax,ymax
0,img_0028,train/images/img_0028.jpg,640,640,3,Jett,292,283,384,445
1,img_1336,train/images/img_1336.jpg,640,640,3,Jett,374,292,397,366
2,img_0996,train/images/img_0996.jpg,640,640,3,Phoenix,294,304,307,349
3,img_0996,train/images/img_0996.jpg,640,640,3,Jett,250,245,267,289
4,img_2159,train/images/img_2159.jpg,640,640,3,Phoenix,94,109,281,639


In [6]:
# Preprocess dataset
data_path = "train.csv"
train_data = pd.read_csv(data_path)

# Set data lists
images = []
labels = []
bboxes = []
image_paths = []

for row in train_data.itertuples():
    # Read image
    image = cv2.imread(row.filepath)

    # Scale bbox coordinates
    xmin = row.xmin / row.width
    xmax = row.xmax / row.width
    ymin = row.ymin / row.height
    ymax = row.ymax / row.height

    # Append data
    images.append(image)
    labels.append(row.label)
    bboxes.append((xmin, ymin, xmax, ymax))
    image_paths.append(row.filepath)

# Train and validation split
split_ratio = 0.8
train_images, val_images, train_labels, val_labels, train_bboxes, val_bboxes, train_image_paths, val_image_paths = train_test_split(
    images, labels, bboxes, image_paths, test_size=1-split_ratio, random_state=seed
)

# Encode labels
le = LabelEncoder()
labels = le.fit_transform(labels)

# Convert to tensors
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)
train_labels = torch.tensor(train_labels, dtype=torch.long)
val_labels = torch.tensor(val_labels, dtype=torch.long)
train_bboxes = torch.tensor(train_bboxes, dtype=torch.float32)
val_bboxes = torch.tensor(val_bboxes, dtype=torch.float32)

RuntimeError: [enforce fail at alloc_cpu.cpp:119] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 12577996800 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Create datasets
train_dataset = ValorantTensorDataset((train_images, train_labels, train_bboxes), transform=transform)
val_dataset = ValorantTensorDataset((val_images, val_labels, val_bboxes), transform=transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)